# SylloGym — Green Agent Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eliot-gtn/syllogym/blob/main/green_agent/green_agent.ipynb)

This notebook demonstrates a **Green Agent** — an autonomous LLM agent interacting with the SylloGym environment.

The agent:
1. Connects to the SylloGym environment (via HuggingFace Space or localhost)
2. Receives a legal rule + case facts
3. Generates a structured chain-of-thought reasoning
4. Submits an answer and receives a reward signal
5. Reports performance across all 10 legal reasoning tasks

You can run this with the **base model** (before training) or a **fine-tuned model** (after GRPO) to see the improvement.

---

## What is a Green Agent?

In the OpenEnv ecosystem, a Green Agent is an agent that:
- Autonomously interacts with an environment without human intervention
- Follows a loop: `observe → reason → act → receive reward`
- Can be evaluated on leaderboards via the [AgentBeats](https://agentbeats.dev) platform

SylloGym's Green Agent demonstrates **legal syllogistic reasoning**: given a rule and facts, it applies deductive logic to reach the correct legal conclusion.

In [ ]:
# ============================================================
# 1. Setup — clone repo and install dependencies
# ============================================================

import os

REPO_NAME = "syllogym"
REPO_URL  = "https://github.com/eliot-gtn/syllogym.git"

if not os.path.exists(f"/content/{REPO_NAME}"):
    !git clone -b develop {REPO_URL} /content/{REPO_NAME}
else:
    !git -C /content/{REPO_NAME} pull

%pip install -q -e /content/{REPO_NAME}/envs/syllogym_env
%pip install -q transformers torch accelerate huggingface_hub

print("Setup complete.")

In [ ]:
# ============================================================
# 2. Configuration
# ============================================================

# SylloGym server URL
SYLLOGYM_URL = "https://farffadet-syllogym-env.hf.space"   # HF Space
# SYLLOGYM_URL = "http://localhost:8000"                    # Local server

# Model to evaluate — swap to compare base vs fine-tuned
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"   # Base model (pre-training)
# MODEL_NAME = "farffadet/syllogym-grpo-Qwen2.5-3B-Instruct"  # Fine-tuned

N_EPISODES  = 100   # Total episodes to run
TASK_MODE   = "mixed"  # "mixed" | "single"
# TASK_NAME = "diversity_1"  # Used only when TASK_MODE="single"

print(f"Model:      {MODEL_NAME}")
print(f"Server:     {SYLLOGYM_URL}")
print(f"Episodes:   {N_EPISODES}")
print(f"Task mode:  {TASK_MODE}")

In [ ]:
# ============================================================
# 3. Load model
# ============================================================

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
model.eval()

device = next(model.parameters()).device
print(f"Model loaded: {MODEL_NAME}")
print(f"Device:       {device}")
if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    print(f"VRAM:         {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ============================================================
# 4. Agent logic
# ============================================================

import re
from syllogym_env import SylloGymEnv, SylloAction

SYSTEM_PROMPT = """You are a legal reasoning expert. You will be given:
1. A legal RULE
2. FACTS of a case
3. A QUESTION

Apply the rule to the facts using deductive reasoning (syllogism):
- Major premise: the rule
- Minor premise: the facts
- Conclusion: your answer

Respond in this exact format:
<reasoning>
[Your step-by-step analysis applying the rule to the facts. Be concise: 2-4 sentences.]
</reasoning>
<answer>[Your answer here]</answer>

Only include your answer word/phrase inside the <answer> tags (no punctuation or explanation)."""


def build_prompt(obs) -> list[dict]:
    """Build the chat messages for a given observation."""
    valid = " | ".join(obs.valid_answers)
    user_content = (
        f"RULE:\n{obs.rule}\n\n"
        f"FACTS:\n{obs.facts}\n\n"
        f"QUESTION: {obs.question}\n\n"
        f"Valid answers: {valid}"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]


def generate(obs, max_new_tokens: int = 400) -> str:
    """Run the model on a SylloGym observation and return the raw response."""
    messages = build_prompt(obs)
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def parse_action(response: str) -> SylloAction:
    """Extract reasoning and answer from the model's structured response."""
    reasoning_match = re.search(r"<reasoning>(.*?)</reasoning>", response, re.DOTALL)
    answer_match    = re.search(r"<answer>(.*?)</answer>",   response, re.DOTALL | re.IGNORECASE)
    reasoning = reasoning_match.group(1).strip() if reasoning_match else response
    answer    = answer_match.group(1).strip()    if answer_match    else response.strip()
    return SylloAction(reasoning=reasoning, answer=answer)


print("Agent ready.")

In [ ]:
# ============================================================
# 5. Run the agent — autonomous episode loop
# ============================================================

from collections import defaultdict

env = SylloGymEnv(base_url=SYLLOGYM_URL)
env.connect()

episode_log   = []   # Full trace of every episode
rewards_by_task = defaultdict(list)

print(f"Running {N_EPISODES} episodes in '{TASK_MODE}' mode...\n")
print(f"{'Ep':>4}  {'Task':<25}  {'Correct':>7}  {'Reward':>7}  {'Format':>7}  Answer")
print("-" * 75)

for ep in range(N_EPISODES):
    # 1. Observe
    result = env.reset(task_mode=TASK_MODE)
    obs    = result.observation

    if not obs.facts or obs.facts.startswith("[Dataset unavailable"):
        print(f"{ep+1:>4}  [dataset unavailable, skipping]")
        continue

    # 2. Reason
    response = generate(obs)
    action   = parse_action(response)

    # 3. Act
    step_result = env.step(action)
    obs_after   = step_result.observation

    reward    = obs_after.reward if obs_after.reward is not None else 0.0
    correct   = reward >= 1.0
    has_fmt   = bool(
        re.search(r"<reasoning>.*?</reasoning>", response, re.DOTALL)
        and re.search(r"<answer>.*?</answer>", response, re.DOTALL | re.IGNORECASE)
    )

    rewards_by_task[obs.task_name].append(reward)
    episode_log.append({
        "episode":        ep + 1,
        "task_name":      obs.task_name,
        "difficulty":     obs.difficulty,
        "facts":          obs.facts,
        "correct_answer": obs.correct_answer,
        "response":       response,
        "predicted":      action.answer,
        "reward":         reward,
        "correct":        correct,
        "has_format":     has_fmt,
    })

    tick = "✓" if correct else "✗"
    fmt  = "yes" if has_fmt else "no"
    pred = action.answer[:15].ljust(15)
    print(f"{ep+1:>4}  {obs.task_name:<25}  {tick:>7}  {reward:>7.2f}  {fmt:>7}  {pred}")

env.disconnect()

print("-" * 75)
print(f"Done. {len(episode_log)} episodes completed.")

In [ ]:
# ============================================================
# 6. Results — per-task breakdown
# ============================================================

print(f"\n{'='*65}")
print(f"GREEN AGENT RESULTS — {MODEL_NAME.split('/')[-1]}")
print(f"{'='*65}")
print(f"{'Task':<30} {'N':>4} {'Accuracy':>9} {'Avg Reward':>11} {'Format':>7}")
print("-" * 65)

all_rewards  = [e["reward"]     for e in episode_log]
all_correct  = [e["correct"]    for e in episode_log]
all_fmt      = [e["has_format"] for e in episode_log]

# Per-task stats
task_stats = {}
for task_name, rewards in sorted(rewards_by_task.items()):
    eps     = [e for e in episode_log if e["task_name"] == task_name]
    acc     = sum(e["correct"]    for e in eps) / len(eps)
    fmt_r   = sum(e["has_format"] for e in eps) / len(eps)
    avg_rew = sum(rewards) / len(rewards)
    task_stats[task_name] = {"accuracy": acc, "avg_reward": avg_rew, "format_rate": fmt_r, "n": len(eps)}
    print(f"{task_name:<30} {len(eps):>4} {acc:>9.1%} {avg_rew:>11.3f} {fmt_r:>7.1%}")

print("-" * 65)
if all_rewards:
    print(f"{'OVERALL':<30} {len(all_rewards):>4} "
          f"{sum(all_correct)/len(all_correct):>9.1%} "
          f"{sum(all_rewards)/len(all_rewards):>11.3f} "
          f"{sum(all_fmt)/len(all_fmt):>7.1%}")

# Show a few interesting examples
print("\n--- Sample correct reasoning ---")
correct_eps = [e for e in episode_log if e["correct"] and e["has_format"]]
if correct_eps:
    ex = correct_eps[0]
    print(f"Task: {ex['task_name']}")
    print(f"Facts: {ex['facts'][:120]}...")
    print(f"Response:\n{ex['response'][:500]}")
    print(f"Reward: {ex['reward']}")

print("\n--- Sample incorrect reasoning ---")
wrong_eps = [e for e in episode_log if not e["correct"]]
if wrong_eps:
    ex = wrong_eps[0]
    print(f"Task: {ex['task_name']}")
    print(f"Facts: {ex['facts'][:120]}...")
    print(f"Correct: {ex['correct_answer']} | Predicted: {ex['predicted']}")
    print(f"Reward: {ex['reward']}")

In [ ]:
# ============================================================
# 7. Visualize results
# ============================================================

try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import numpy as np

    tasks   = list(task_stats.keys())
    accs    = [task_stats[t]["accuracy"]   for t in tasks]
    rewards = [task_stats[t]["avg_reward"] for t in tasks]
    ns      = [task_stats[t]["n"]          for t in tasks]

    # Color bars green/red by accuracy
    colors = ["#2ecc71" if a >= 0.7 else "#e74c3c" if a < 0.5 else "#f39c12" for a in accs]

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Accuracy chart
    bars = axes[0].barh(tasks, accs, color=colors, edgecolor="white", linewidth=0.5)
    axes[0].axvline(x=0.5, color="gray", linestyle="--", alpha=0.6, label="Random (50%)")
    axes[0].set_xlabel("Accuracy")
    axes[0].set_title(f"Task Accuracy — {MODEL_NAME.split('/')[-1]}")
    axes[0].set_xlim(0, 1.05)
    for bar, acc, n in zip(bars, accs, ns):
        axes[0].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                     f"{acc:.0%} (n={n})", va="center", fontsize=8)
    axes[0].legend()

    # Reward chart
    bars2 = axes[1].barh(tasks, rewards, color=colors, edgecolor="white", linewidth=0.5)
    axes[1].set_xlabel("Average Reward")
    axes[1].set_title(f"Average Reward — {MODEL_NAME.split('/')[-1]}")
    axes[1].set_xlim(0, 1.4)
    axes[1].axvline(x=1.0, color="blue", linestyle="--", alpha=0.4, label="Correct answer threshold")
    axes[1].legend()

    # Legend
    patches = [
        mpatches.Patch(color="#2ecc71", label=">= 70% accuracy"),
        mpatches.Patch(color="#f39c12", label="50–70% accuracy"),
        mpatches.Patch(color="#e74c3c", label="< 50% accuracy"),
    ]
    fig.legend(handles=patches, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.08))

    plt.suptitle(f"SylloGym Green Agent — {N_EPISODES} episodes", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig("green_agent_results.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Chart saved to green_agent_results.png")

except ImportError:
    print("matplotlib not available — skipping chart")